[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/padillacm/IA-practice/blob/master/tokenization-embeddings-course/00_intro/00_intro.ipynb)

# Módulo 00 · Intro, entorno e hilo conductor
**Tokenización y Embeddings: de BPE a los modelos modernos**

| | |
|---|---|
| **Nivel** | 🟢 Básico |
| **Duración** | ≈ 2 h en total (lección ≈ 1 h + proyecto ≈ 1 h) |
| **GPU / coste** | Ninguna: todo corre en CPU (0 unidades de cómputo de Colab Pro) |
| **Prerrequisitos** | Ninguno dentro del curso. Se asume Python sólido, PyTorch, Transformers y álgebra lineal/probabilidad |

> **Estado del arte de referencia: octubre de 2026.** Toda cifra sujeta a cambio lleva fecha; las técnicas se etiquetan 🟢 sólido / 🟡 emergente / 🔴 experimental.

## Objetivos de aprendizaje
Al terminar este módulo podrás:

1. **Describir** el pipeline completo *texto → bytes → tokens → ids → embeddings → índice → ranking* y situar cada módulo del curso en él.
2. **Distinguir** token (unidad discreta de un vocabulario) de embedding (vector aprendido asociado a una unidad o a un texto) con un ejemplo numérico.
3. **Configurar** el entorno de Colab del curso (semilla, `device`, `FAST_DEV_RUN`) y **reemplazar** correctamente las celdas 🎨/🎬 de Gemini por imágenes/videos en `assets/`.
4. **Cargar** el corpus Políglota con `carga_corpus()` (cadena oficial → espejo → sintético) e **interpretar** su esquema.
5. **Implementar** nDCG@k, Recall@k y MRR@k con el tratamiento correcto de empates, duplicados y consultas sin relevantes.
6. **Construir** un baseline BM25 con índice invertido, **validarlo** contra `rank_bm25` y **medir** su nDCG@10 por idioma con un intervalo de confianza bootstrap.
7. **Diagnosticar** por qué un baseline léxico falla (idioma, vocabulario, tokenización) y cómo eso motiva los módulos siguientes.

## 1. El mapa del curso
El curso tiene **21 módulos (00–20, ≈ 104 h)** y dos bloques: primero entendemos cómo un texto se convierte en **tokens** (bloque A) y luego cómo los tokens y los textos se convierten en **embeddings** que se pueden indexar y buscar (bloque B). Todo converge en **Políglota**, un buscador semántico multilingüe (ES/EN/código).

```text
 BLOQUE A · Texto y tokenización                         BLOQUE B · Embeddings
 00 Intro ─ 01 Unicode/bytes ─ 02 BPE ─ 03 WordPiece/    09 Capa de embedding ─ 10 Contextuales ─ 11 Contrastivo
 Unigram/SentencePiece ─ 04 Diseño ─ 05 Patologías ─     12 Embedders modernos ─ 13 Dense/sparse/late ─ 14 ANN
 06 Bytes/BLT ─ 07 H-Net/transfer ─ 08 Multimodal/audio  15 Evaluación ─ 16 Multimodal ─ 17 Interpretabilidad
                                                         18 Producción ─ 19 Playbook ─ 20 Capstone
```

In [ ]:
import matplotlib.pyplot as plt

AZUL, NARANJA, VERDE, GRIS, ROJO = "#1F6FEB", "#F28E2B", "#2CA02C", "#6B7280", "#D62728"
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.25, "axes.spines.top": False,
                     "axes.spines.right": False, "font.size": 10, "axes.titlesize": 11})

# Horas por módulo (SYLLABUS del curso) coloreadas por bloque
mods = [("00 Intro", 2), ("01 Unicode/bytes", 4), ("02 BPE", 5), ("03 WordPiece/Unigram/SP", 5), ("04 Diseño tokenizers", 5),
        ("05 Patologías", 5), ("06 Bytes/BLT", 5), ("07 H-Net/transfer", 6), ("08 Multimodal/audio", 6),
        ("09 Capa de embedding", 5), ("10 Contextuales", 5), ("11 Contrastivo", 6), ("12 Embedders modernos", 5),
        ("13 Dense/sparse/late", 5), ("14 ANN", 5), ("15 Evaluación", 5), ("16 Multimodal", 5), ("17 Interpretabilidad", 5),
        ("18 Producción", 5), ("19 Playbook", 4), ("20 Capstone", 6)]
colores = [AZUL] * 9 + [NARANJA] * 9 + [VERDE] * 3
fig, ax = plt.subplots(figsize=(8, 6))
ax.barh([m for m, _ in mods][::-1], [h for _, h in mods][::-1], color=colores[::-1])
ax.set_xlabel("horas estimadas")
ax.set_title(f"Horas por módulo (total ≈ {sum(h for _, h in mods)} h)")
from matplotlib.patches import Patch
ax.legend(handles=[Patch(color=AZUL, label="A · Tokenización"), Patch(color=NARANJA, label="B · Embeddings"),
                   Patch(color=VERDE, label="Producción y cierre")], loc="lower right")
plt.tight_layout(); plt.show()

> 👀 **Qué debes observar.** Los módulos de tokenización (00–08) pesan ≈ 43 h, los de embeddings (09–17) ≈ 46 h y producción/cierre (18–20) ≈ 15 h: **tokenizar bien es cerca de la mitad del trabajo técnico**, y no es un detalle menor porque todo lo que viene después (costos, calidad multilingüe, robustez) depende de ello.
> Los módulos 02, 06, 07, 11 y 13 son los más densos: reserva sesiones largas.

> ### 🎨 GEMINI-IMAGEN · ID: `M00-IMG-01` · ⚠️ REEMPLAZAR ESTA CELDA
> **Mapa del curso en dos bloques**
>
> **Qué muestra:** Un mapa visual de los 21 módulos agrupados en bloque A (tokenización) y bloque B (embeddings) con el hilo conductor Políglota creciendo por debajo. Sirve para orientarte y volver a él al cambiar de módulo.
> **Formato:** PNG 16:9, 1920x1080 | **Texto alternativo:** Mapa de 21 módulos en dos bloques: tokenización (00–08) y embeddings (09–17) más producción y capstone (18–20)
>
> **Prompt para Gemini (copiar y pegar):**
> ```text
> Diagrama técnico limpio, fondo blanco, estilo de infografía de curso, 16:9. Paleta sobria: azul #1F6FEB, naranja #F28E2B, verde #2CA02C, gris #6B7280, tipografía sans-serif legible, sin ilustraciones decorativas.
> Composición: un camino horizontal de izquierda a derecha con 21 estaciones numeradas, dividido en dos filas.
> Fila superior (franja azul, título «BLOQUE A · Texto y tokenización»), de izquierda a derecha, 9 cajas redondeadas con estos textos EXACTOS:
> «00 Intro», «01 Unicode y bytes», «02 BPE desde cero», «03 WordPiece, Unigram, SentencePiece», «04 Diseño de tokenizers», «05 Patologías y robustez», «06 Bytes y BLT», «07 H-Net y tokenizer transfer», «08 Tokens multimodales y audio».
> Fila inferior (franja naranja, título «BLOQUE B · Embeddings»), de izquierda a derecha, 9 cajas con estos textos EXACTOS:
> «09 Capa de embedding», «10 Contextuales y de oraciones», «11 Aprendizaje contrastivo», «12 Embedders modernos», «13 Dense, sparse y late interaction», «14 Indexación y ANN», «15 Evaluación», «16 Embeddings multimodales», «17 Interpretabilidad y geometría».
> A la derecha, una franja verde vertical con 3 cajas apiladas: «18 Producción», «19 Playbook de élite», «20 Capstone: Políglota completo».
> Una flecha gris conecta cada caja con la siguiente; una flecha curva une «08» (fin de la fila superior) con «09» (inicio de la fila inferior).
> Debajo de todo, una barra horizontal gris con degradado hacia verde y la etiqueta «Políglota: buscador semántico multilingüe (ES/EN/código) crece en cada módulo».
> Título superior: «Tokenización y Embeddings: de BPE a los modelos modernos».
> Evita: texto ilegible o inventado, más o menos de 21 módulos, flechas ambiguas, iconos decorativos, sombras exageradas.
> ```
>
> **Cómo insertarlo:** guarda el archivo como `assets/M00-IMG-01.png` en la carpeta del módulo y reemplaza esta celda por: `![Mapa de 21 módulos en dos bloques: tokenización (00–08) y embeddings (09–17) más producción y capstone (18–20)](assets/M00-IMG-01.png)`
>
> **Mientras tanto (placeholder):**
> ```text
> BLOQUE A · Texto y tokenización
> [00]→[01]→[02]→[03]→[04]→[05]→[06]→[07]→[08]
>                                          ↓
> BLOQUE B · Embeddings
> [09]→[10]→[11]→[12]→[13]→[14]→[15]→[16]→[17] → [18]→[19]→[20]
> ══════════ Políglota crece en cada módulo ═══════════►
> ```

> 👀 **Qué debes observar.** Cuando termines cada módulo vuelve a este mapa y ubica qué pieza de Políglota acabas de construir. Fíjate en el salto 08 → 09: pasamos de «cómo se parte el texto» a «qué vector le corresponde»; el puente entre ambos bloques es la **matriz de embeddings** (módulo 09).

## 2. Cómo usar este curso (Colab, `FAST_DEV_RUN`, celdas Gemini y `assets/`)

**Colab.** Abre cada notebook con el badge «Open In Colab». T4 es la GPU por defecto (Colab Pro); algunos módulos (07, 11, 16) recomiendan L4/A100 y lo indican en su cabecera con una estimación de unidades. **Este módulo y los tres primeros corren enteros en CPU.**

**`FAST_DEV_RUN`.** Toda celda pesada lee un flag `FAST_DEV_RUN = True` por defecto: con él, el notebook corre en minutos con datos diminutos (sirve para depurar y para entender). Cuando quieras resultados «de verdad», pon `FAST_DEV_RUN = False` (idealmente en GPU).

**Semilla y `device`.** Siempre `seed = 42` y detección de GPU. Los números que veas en tu pantalla dependen de la versión de las librerías y del corpus (real o sintético): por eso las rúbricas de los proyectos se calibran con lo que **mide la solución de referencia**, no con lo deseado.

**Celdas Gemini (🎨 imagen / 🎬 video).** Algunos diagramas y animaciones se generan **con Gemini**, no con código (los resultados con datos *nunca* van en celdas Gemini). Cada una es un bloque como el de abajo con: un **ID** único (`M00-IMG-01`), un **prompt completo y autocontenido** que copias y pegas en Gemini sin editar, el **nombre de archivo** que debes usar y un **placeholder** que se ve mientras no la reemplaces. Pasos:

1. Copia el prompt del bloque (la parte `text`) en Gemini y genera la imagen/video.
2. Guarda el archivo en la carpeta `assets/` de ese módulo con el nombre que indica la celda (p. ej. `assets/M00-IMG-01.png`).
3. Reemplaza la celda por la línea `![texto alternativo](assets/M00-IMG-01.png)` (o la etiqueta `<video …>` para video).

Aunque no las reemplaces, la lección se entiende con el placeholder. El índice de todos los recursos del curso está en `extras/GEMINI_ASSETS.md`.

> **📚 Para profundizar.** Si trabajas en Colab y quieres que `assets/` persista, clona el repositorio en Drive o sube los archivos a la carpeta del módulo; las rutas del curso son relativas al notebook.

In [ ]:
# ── Entorno del curso (celda de arranque estándar) ──
!pip install -q numpy pandas matplotlib seaborn tiktoken rank_bm25 regex

In [ ]:
import os, random, sys, time
import numpy as np

SEED = 42
random.seed(SEED); np.random.seed(SEED)
FAST_DEV_RUN = True   # True: subconjuntos diminutos, minutos en CPU. False: escala completa.

try:
    import torch
    torch.manual_seed(SEED); torch.set_num_threads(1)
    device = "cuda" if torch.cuda.is_available() else "cpu"
except ImportError:      # este módulo no necesita torch
    device = "cpu"
print(f"python {sys.version.split()[0]} | numpy {np.__version__} | device = {device} | FAST_DEV_RUN = {FAST_DEV_RUN}")

## 3. Intuición primero: del texto a un resultado de búsqueda

Un buscador moderno es una cadena de transformaciones. **Cada flecha pierde o crea información**, y la mayoría de los errores «misteriosos» de un sistema de retrieval nacen en las primeras flechas, antes de que el modelo vea nada.

```text
texto ──► bytes ──► tokens ──► ids ──► embeddings ──► (Transformer) ──► vector del texto ──► índice ──► ranking
  M01       M01       M02-08    M02-08    M09          M10-12              M10-12              M14       M13,15
"café"   63 61 66  c·a·f·é     [id…]     [vector…]                       [768 números…]
         c3 a9
```

* **Bytes (módulo 01).** El ordenador no ve letras: ve bytes. «é» ocupa 2 bytes en UTF-8; un ideograma chino, 3; un emoji, 4. Esto condiciona el costo de todo lo demás.
* **Tokens (módulos 02–08).** Un *tokenizer* parte el texto en unidades de un vocabulario fijo (≈ 32 000–262 000 entradas en modelos actuales; p. ej. Gemma 3 usa 262 144). Es **discreto y determinista**: no aprende con gradientes.
* **Embeddings (módulos 09–12).** Cada token (y cada texto completo) se asocia a un **vector denso** que *sí* se aprende. Textos parecidos → vectores cercanos.
* **Índice y ranking (módulos 13–15).** Buscar es encontrar los vectores más cercanos a la consulta (o combinar con señales léxicas como BM25).

> **Analogía con lo que ya sabes (recomendación/IR).** El tokenizer es el *parser de términos* de un motor de búsqueda; el embedding es el *vector de ítem* de un sistema de recomendación; el índice ANN es el *candidate generator* de dos torres. Y el vocabulario es a un LLM lo que el catálogo de ítems a un recsys: lo que no esté en el catálogo, no se puede representar directamente.

> ### 🎨 GEMINI-IMAGEN · ID: `M00-IMG-02` · ⚠️ REEMPLAZAR ESTA CELDA
> **Pipeline: del texto al ranking**
>
> **Qué muestra:** El recorrido completo de una consulta: texto, bytes, tokens, ids, embeddings, vector de la consulta, búsqueda en el índice y lista ordenada. Con un ejemplo real tokenizado con GPT-2.
> **Formato:** PNG 16:9, 1920x1080 | **Texto alternativo:** Pipeline de 7 etapas: texto, bytes, tokens, ids, embeddings, vector del texto, índice y ranking
>
> **Prompt para Gemini (copiar y pegar):**
> ```text
> Diagrama técnico limpio de un pipeline, fondo blanco, 16:9, flechas gruesas de izquierda a derecha. Paleta: azul #1F6FEB para texto y tokens, naranja #F28E2B para vectores, verde #2CA02C para el índice y los resultados, gris #6B7280 para anotaciones. Tipografía sans-serif legible, sin ilustraciones decorativas.
> Composición: 7 etapas en una fila, cada una como una caja redondeada con título arriba y contenido debajo. Los textos son EXACTOS:
> 1) Título «Texto». Contenido: «Tokenizers turn text into numbers.»
> 2) Título «Bytes (UTF-8)». Contenido: «54 6F 6B 65 6E 69 7A 65 72 73 …» y debajo en gris pequeño «1 byte por letra ASCII».
> 3) Título «Tokens». Contenido: siete cajitas azules consecutivas con estos textos exactos: «Token», «izers», « turn», « text», « into», « numbers», «.»
> 4) Título «IDs (vocabulario GPT-2)». Contenido: siete números exactos bajo cada token, en este orden: «30642», «11341», «1210», «2420», «656», «3146», «13».
> 5) Título «Embeddings». Contenido: siete barras verticales pequeñas naranjas (un vector por token) con la etiqueta «un vector por token (se aprende)».
> 6) Título «Vector del texto». Contenido: una sola barra naranja alargada con la etiqueta «un vector por texto (pooling)».
> 7) Título «Índice y ranking». Contenido: un pequeño espacio 2D con puntos grises y tres puntos verdes cercanos a una estrella naranja (la consulta), y a la derecha una lista «1. doc A», «2. doc B», «3. doc C».
> Encima de las etapas, una cinta con los nombres de los bloques del curso: sobre las etapas 2-4 «Bloque A · Tokenización», sobre las etapas 5-7 «Bloque B · Embeddings».
> Título superior: «De texto a resultado de búsqueda».
> Evita: números distintos de los indicados, tokens inventados, texto ilegible, flechas que se crucen, elementos decorativos o personas.
> ```
>
> **Cómo insertarlo:** guarda el archivo como `assets/M00-IMG-02.png` en la carpeta del módulo y reemplaza esta celda por: `![Pipeline de 7 etapas: texto, bytes, tokens, ids, embeddings, vector del texto, índice y ranking](assets/M00-IMG-02.png)`
>
> **Mientras tanto (placeholder):**
> ```text
> Texto            Bytes        Tokens                                   IDs                              Embeddings     Texto→vector   Índice
> "Tokenizers  →  54 6F 6B 65 → [Token][izers][ turn][ text][ into][ numbers][.] → 30642 11341 1210 2420 656 3146 13 → ▮▮▮▮▮▮▮ (7 vectores) → ▮ (1 vector) → ★ ← ● ● ●
>  turn text ..."   6E 69 7A …
>  (Bloque A: tokenización ────────────────────────────)   (Bloque B: embeddings ─────────────────────────────────────)
> ```

> 👀 **Qué debes observar.** Con el recurso insertado, sigue los **siete** IDs de izquierda a derecha: son exactamente los que obtendrás con `tiktoken` y el vocabulario GPT-2 en la celda siguiente. Fíjate en que la frase en inglés tarda **7 tokens**; veremos que la misma idea en español cuesta bastante más (el «impuesto de tokens»).

## 4. Qué es un token (visión global)

Un **token** es una unidad *discreta* de un vocabulario fijo $\mathcal{V}$ (un conjunto finito de cadenas o de secuencias de bytes) a la que se le asigna un **entero** (su id). Un tokenizer es una función $\text{tok}: \Sigma^* \to \mathcal{V}^*$ (de cadenas a secuencias de tokens) y su inversa aproximada, $\text{detok}$.

Un token **no es una palabra**. Veamos la misma frase cortada de cuatro maneras: por palabras, por caracteres, por bytes y por el tokenizer de GPT-2 (byte-level BPE, que implementarás en el módulo 02).

In [ ]:
import hashlib
import os
import tempfile
import urllib.request


def get_gpt2_encoding():
    """tiktoken 'gpt2'. 1) oficial (openaipublic); 2) espejo en GitHub (graykode/gpt-2-Pytorch) verificado con los
    mismos SHA-256 que exige tiktoken; 3) None (las celdas que lo usan se omiten con un aviso)."""
    import tiktoken
    try:
        return tiktoken.get_encoding("gpt2")
    except Exception as e:  # noqa: BLE001
        print(f"[tiktoken] descarga oficial no disponible ({type(e).__name__}); pruebo el espejo de GitHub...")
    try:
        import tiktoken_ext.openai_public as op
        from tiktoken.load import data_gym_to_mergeable_bpe_ranks
        base = "https://raw.githubusercontent.com/graykode/gpt-2-Pytorch/master/GPT2/"
        sha = {"vocab.bpe": "1ce1664773c50f3e0cc8842619a93edc4624525b728b188a9e0be33b7726adc5",
               "encoder.json": "196139668be63f3b5d6574427317ae82f612a97c5d1cdaf36ed2256dbf636783"}
        paths = {}
        for name, h in sha.items():
            data = urllib.request.urlopen(base + name, timeout=60).read()
            assert hashlib.sha256(data).hexdigest() == h, f"hash de {name} no coincide"
            paths[name] = os.path.join(tempfile.gettempdir(), "gpt2_" + name)
            with open(paths[name], "wb") as f:
                f.write(data)
        ranks = data_gym_to_mergeable_bpe_ranks(vocab_bpe_file=paths["vocab.bpe"], encoder_json_file=paths["encoder.json"])
        print("[tiktoken] GPT-2 cargado desde el espejo de GitHub (hashes verificados)")
        return tiktoken.Encoding("gpt2", pat_str=op.r50k_pat_str, mergeable_ranks=ranks,
                                 special_tokens={"<|endoftext|>": 50256})
    except Exception as e:  # noqa: BLE001
        print(f"[tiktoken] sin GPT-2 ({type(e).__name__}: {str(e)[:80]}). Las comparaciones con tiktoken se omiten.")
        return None

In [ ]:
frases = {
    "ES": "Los tokenizadores convierten texto en números.",
    "EN": "Tokenizers turn text into numbers.",
    "código": "def añadir_ñu(x): return x+1",
}
enc = get_gpt2_encoding()
filas = []
for nombre, s in frases.items():
    ids = enc.encode(s) if enc else []
    filas.append({"idioma": nombre, "palabras (split)": len(s.split()), "caracteres": len(s),
                  "bytes UTF-8": len(s.encode("utf-8")), "tokens GPT-2": len(ids) if enc else float("nan")})
import pandas as pd
df_tok = pd.DataFrame(filas).set_index("idioma")
display(df_tok)
if enc:
    ids = enc.encode(frases["ES"])
    print("ES →", [enc.decode_single_token_bytes(i).decode("utf-8", "replace") for i in ids])
    ids = enc.encode(frases["EN"])
    print("EN →", [enc.decode_single_token_bytes(i).decode("utf-8", "replace") for i in ids], ids)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.8))
df_tok.plot.bar(ax=ax, color=[GRIS, AZUL, NARANJA, VERDE], rot=0, width=0.8)
ax.set_ylabel("unidades por frase"); ax.set_title("La misma idea, cuatro granularidades")
plt.tight_layout(); plt.show()

> 👀 **Qué debes observar.** 1. **Palabras < tokens < caracteres ≈ bytes** en inglés, pero en español el número de tokens se acerca más al de caracteres: el vocabulario de GPT-2 se construyó mayoritariamente con inglés.
> 2. «ñ» y «ú» ocupan **2 bytes** cada una: bytes ≠ caracteres (módulo 01).
> 3. El código también se fragmenta: `_`, `(`, `)` y los espacios consumen tokens. Más tokens = más costo y menos contexto útil.

> ### 🧪 Mini-experimento: ¿cuánto cuesta un idioma?
> La relación **tokens / palabra** se llama *fertilidad*. En esta frase el español rinde peor que el inglés. En el módulo 04 la medirás con corpus paralelos (Flores) y en el 02 verás **por qué** ocurre (qué texto se usó para aprender las fusiones). Dato de contexto: Petrov et al. (2023) y Ahia et al. (2023) documentan que la misma frase traducida puede costar hasta 15 veces más tokens según el idioma y el tokenizer (ver referencias).

## 5. Qué es un embedding (visión global)

Un **embedding** es un vector denso $\mathbf{e}\in\mathbb{R}^d$ que representa una unidad (un token, una oración, un documento, una imagen…) de modo que la **geometría refleje similitud**: unidades parecidas ⇒ vectores cercanos.

Hay **dos usos** que no debes confundir:

| | Embedding de token (capa de embedding) | Embedding de texto (sentence/document embedding) |
|---|---|---|
| Entrada | un id entero $i \in \{0,\dots,|\mathcal{V}|-1\}$ | una secuencia de tokens |
| Salida | fila $i$ de una matriz $E \in \mathbb{R}^{|\mathcal{V}|\times d}$ | un vector que resume todo el texto |
| Cómo se obtiene | búsqueda en tabla (*lookup*) | pasar por un Transformer y hacer *pooling* |
| Módulos | 09 | 10, 11, 12 |

**Lookup = producto con one-hot.** Si $\mathbf{x}_i\in\{0,1\}^{|\mathcal{V}|}$ es el one-hot del id $i$, entonces $\mathbf{x}_i^\top E = E_{i,:}$. Es una multiplicación matricial que nadie hace de verdad (se indexa), pero explica por qué los gradientes solo actualizan las filas de los tokens presentes.

**Similitud.** Para comparar vectores se usa casi siempre el **coseno**:
$$\cos(\mathbf{u},\mathbf{v})=\frac{\mathbf{u}^\top\mathbf{v}}{\lVert\mathbf{u}\rVert\,\lVert\mathbf{v}\rVert}\in[-1,1]$$
donde $\mathbf{u}^\top\mathbf{v}=\sum_k u_kv_k$ es el producto punto y $\lVert\cdot\rVert$ la norma euclídea. Si los vectores están normalizados ($\lVert\mathbf{u}\rVert=1$), coseno = producto punto.

A continuación, un **juguete hecho a mano** (los vectores NO están aprendidos; los escribimos para fijar la idea): 6 palabras en un espacio de 2 dimensiones con un eje «animal» y otro «vehículo», incluyendo traducciones ES/EN.

In [ ]:
palabras = ["gato", "cat", "perro", "coche", "car", "camión"]
V = np.array([[0.90, 0.10], [0.88, 0.15], [0.80, 0.20], [0.10, 0.90], [0.12, 0.88], [0.20, 0.80]])  # (6, 2) a mano
Vn = V / np.linalg.norm(V, axis=1, keepdims=True)
cos = Vn @ Vn.T

fig, axs = plt.subplots(1, 2, figsize=(10, 4))
for (x, y), p, c in zip(V, palabras, [AZUL, AZUL, AZUL, NARANJA, NARANJA, NARANJA]):
    axs[0].annotate("", xy=(x, y), xytext=(0, 0), arrowprops=dict(arrowstyle="->", color=c, lw=1.6))
    axs[0].text(x + 0.02, y + 0.02, p, color=c)
axs[0].set_xlim(0, 1.1); axs[0].set_ylim(0, 1.1); axs[0].set_xlabel("eje «animal»"); axs[0].set_ylabel("eje «vehículo»")
axs[0].set_title("Vectores de juguete en 2D")
im = axs[1].imshow(cos, vmin=0, vmax=1, cmap="Blues")
axs[1].set_xticks(range(6), palabras, rotation=45); axs[1].set_yticks(range(6), palabras)
for i in range(6):
    for j in range(6):
        axs[1].text(j, i, f"{cos[i, j]:.2f}", ha="center", va="center", fontsize=8, color="white" if cos[i, j] > 0.6 else "black")
axs[1].set_title("Similitud coseno"); axs[1].grid(False); plt.colorbar(im, ax=axs[1], fraction=0.046)
plt.tight_layout(); plt.show()
print("cos(gato, cat) =", round(float(cos[0, 1]), 3), "| cos(gato, coche) =", round(float(cos[0, 3]), 3))

> 👀 **Qué debes observar.** Dos bloques cuadrados en la matriz: animales por un lado, vehículos por otro, y **las traducciones (gato/cat, coche/car) son vecinas**. Eso es lo que *aprenden* los embedders multilingües (módulos 10–12). Un buscador léxico como BM25, en cambio, ve «gato» y «cat» como cadenas distintas (lo comprobaremos en la sección 9).

In [ ]:
# De token id a vector: lookup en una matriz E (aleatoria: aún no hay aprendizaje)
rng = np.random.default_rng(SEED)
vocab_juguete = {"<pad>": 0, "el": 1, "gato": 2, "duerme": 3, "coche": 4}
E = rng.normal(0, 1, size=(len(vocab_juguete), 3))          # |V| x d
ids = [vocab_juguete[t] for t in ["el", "gato", "duerme"]]
print("ids:", ids)
print("lookup E[ids] =\n", E[ids].round(2))
onehot = np.eye(len(vocab_juguete))[ids]                      # (3, |V|)
assert np.allclose(onehot @ E, E[ids])                        # one-hot @ E == indexar
print("✔ one-hot @ E == E[ids]")

> **💡 Intuición.** La matriz $E$ de arriba es aleatoria, así que sus filas **no significan nada todavía**. En un LLM real se aprende junto con el resto del modelo (módulo 09); en un *embedder* de texto se aprende con aprendizaje contrastivo (módulo 11). Lo único que fija este módulo es la **forma** del problema: tokens discretos → vectores continuos.

## 6. El hilo conductor: Políglota

**Contexto.** Eres ML engineer en **Políglota Labs**, una startup que construye un buscador semántico para documentación técnica en **español, inglés y código**. En cada módulo añades una pieza; en el módulo 20 las integras.

| Etapa | Módulos | Pieza de Políglota |
|---|---|---|
| Corpus + baseline | **00** | `carga_corpus()`, métricas, **BM25** (este módulo) |
| Tokenizer propio | 01–05 | auditoría Unicode, BPE, WordPiece/Unigram, diseño de vocabulario, robustez |
| Alternativas | 06–08 | bytes/patches, H-Net, tokenizer transfer, tokens multimodales |
| Embeddings | 09–12 | capa de embedding, pooling, entrenamiento contrastivo, embedders modernos |
| Recuperación híbrida | 13–14 | dense + sparse + late interaction, índice ANN |
| Evaluación y más | 15–17 | banco de evaluación, multimodal, interpretabilidad |
| Producto | 18–20 | servicio, decisiones de diseño, capstone |

> ### 🎨 GEMINI-IMAGEN · ID: `M00-IMG-03` · ⚠️ REEMPLAZAR ESTA CELDA
> **Arquitectura final de Políglota**
>
> **Qué muestra:** La arquitectura objetivo del buscador (la que construirás por piezas): ingesta y normalización, tokenizer propio, embedder, índice híbrido, reranker y evaluación; cada pieza marcada con el módulo donde se construye.
> **Formato:** PNG 16:9, 1920x1080 | **Texto alternativo:** Arquitectura de Políglota con carriles de indexación y consulta, índice híbrido, reranker, evaluación y servicio
>
> **Prompt para Gemini (copiar y pegar):**
> ```text
> Diagrama de arquitectura limpio, fondo blanco, 16:9. Paleta: azul #1F6FEB (texto y tokens), naranja #F28E2B (vectores y modelos), verde #2CA02C (índice y servicio), gris #6B7280 (anotaciones). Tipografía sans-serif legible, flechas claras, sin ilustraciones decorativas.
> Composición de izquierda a derecha en dos carriles.
> Carril superior «Indexación (offline)»: cajas conectadas por flechas con estos textos EXACTOS, de izquierda a derecha:
> «Corpus ES/EN/código [M00]» → «Normalización Unicode [M01]» → «Tokenizer BPE propio [M02-M05]» → «Embedder (dense) [M09-M12]» → «Índice ANN + índice BM25 [M13-M14]».
> Carril inferior «Consulta (online)»: «Consulta del usuario» → «Normalización + tokenizer» → «Embedder + BM25» → «Fusión híbrida (RRF) [M13]» → «Reranker cross-encoder [M13]» → «Top-10 resultados».
> Una flecha vertical de la caja «Índice ANN + índice BM25» del carril superior hacia la caja «Fusión híbrida (RRF)» del inferior.
> A la derecha, una columna vertical verde con dos cajas: «Evaluación: nDCG@10, Recall@k, MRR [M15]» y «Servicio, caché y monitoreo [M18]».
> Una caja punteada gris en la esquina inferior izquierda: «Opcional: imágenes y PDFs [M16]».
> Título superior: «Políglota: buscador semántico multilingüe».
> Evita: añadir cajas o módulos que no estén en esta lista, texto ilegible, iconos decorativos, flechas ambiguas.
> ```
>
> **Cómo insertarlo:** guarda el archivo como `assets/M00-IMG-03.png` en la carpeta del módulo y reemplaza esta celda por: `![Arquitectura de Políglota con carriles de indexación y consulta, índice híbrido, reranker, evaluación y servicio](assets/M00-IMG-03.png)`
>
> **Mientras tanto (placeholder):**
> ```text
> INDEXACIÓN (offline)
> [Corpus M00] → [Normalización M01] → [Tokenizer BPE M02-05] → [Embedder M09-12] → [Índice ANN + BM25 M13-14]
>                                                                                           │
> CONSULTA (online)                                                                         ▼
> [Consulta] → [Norm + tokenizer] → [Embedder + BM25] ──────────────► [Fusión RRF M13] → [Reranker M13] → [Top-10]
>                                                                    [Evaluación M15] [Servicio M18]
> ```

> 👀 **Qué debes observar.** Cada caja lleva el módulo donde la construyes. Nota que el **tokenizer** y la **normalización** están *antes* del embedder y del BM25: si ahí se pierde información (una tilde, un emoji, un espacio), ningún modelo posterior la recupera.

## 7. El corpus Políglota: `carga_corpus()`

Todos los módulos usan **el mismo contrato de datos**. La función `carga_corpus(fast_dev)` devuelve un diccionario:

```python
{
  "docs":    [{"id": str, "lang": "es" | "en" | "code", "text": str}, ...],
  "queries": [{"qid": str, "lang": "es" | "en" | "code", "text": str}, ...],
  "qrels":   {qid: {doc_id: 1}},          # relevancia (>0 = relevante)
  "source":  "real" | "synthetic",
}
```

**Cadena de descarga (oficial → espejo → sintético):**

| Parte | 1) oficial | 2) espejo | 3) último recurso |
|---|---|---|---|
| Wikipedia ES/EN | HF `wikimedia/wikipedia` (snapshot `20231101`, *streaming*) | API pública de Wikipedia (artículos aleatorios, solo la introducción) | texto técnico **sintético** ES/EN (avisa en rojo) |
| Código Python | CodeSearchNet en HF (varios repos por cambios de formato de `datasets`) | funciones **reales** de la biblioteca estándar de Python con su docstring | — |

**Mini golden set.** Cada consulta es una búsqueda de *«ítem conocido»*: en Wikipedia, la consulta es el **título** del artículo (que no aparece en el `text`) y el único relevante es ese artículo; en código, la consulta es la **primera frase del docstring** y el relevante es la función (con el docstring ya eliminado). Es barato y reproducible, pero tiene un sesgo que discutiremos en «Secretos de la élite».

> ⚠️ Si `source == "synthetic"`, los números absolutos **no son comparables** con datos reales: sirven para depurar el código, no para sacar conclusiones sobre Wikipedia.

Cada notebook del curso es autocontenido: copia estas celdas (la implementación canónica vive en el repositorio, `_tools/builders/poliglota_src/`).

In [ ]:
# ── Celda 1/3 de carga_corpus · datos del corpus SINTÉTICO (último recurso) ──
# ── Datos para el corpus SINTÉTICO de Políglota (último recurso si no hay red / Hugging Face) ──
# (artículo+concepto ES, artículo+concepto EN, definición ES, definición EN)
CONCEPTOS = [
    ("el índice invertido", "the inverted index", "una estructura que asocia cada término con la lista de documentos que lo contienen", "a structure that maps each term to the list of documents that contain it"),
    ("la caché", "the cache", "una memoria rápida que guarda resultados recientes para no recalcularlos", "a fast memory that stores recent results so they are not recomputed"),
    ("la base de datos relacional", "the relational database", "un sistema que organiza los datos en tablas relacionadas mediante claves", "a system that organizes data into tables related through keys"),
    ("el balanceador de carga", "the load balancer", "un componente que reparte las peticiones entre varios servidores", "a component that spreads requests across several servers"),
    ("el contenedor Docker", "the Docker container", "un proceso aislado que empaqueta una aplicación con sus dependencias", "an isolated process that packages an application with its dependencies"),
    ("el control de versiones", "version control", "un sistema que registra los cambios de los archivos a lo largo del tiempo", "a system that records changes to files over time"),
    ("la API REST", "the REST API", "una interfaz que expone recursos mediante URL y verbos HTTP", "an interface that exposes resources through URLs and HTTP verbs"),
    ("la autenticación con tokens", "token authentication", "un mecanismo que identifica al usuario con un testigo firmado en cada petición", "a mechanism that identifies the user with a signed token on every request"),
    ("la expresión regular", "the regular expression", "un patrón que describe un conjunto de cadenas de texto", "a pattern that describes a set of text strings"),
    ("el hilo de ejecución", "the thread", "una secuencia de instrucciones que comparte memoria con otros hilos del proceso", "a sequence of instructions that shares memory with other threads of the process"),
    ("el recolector de basura", "the garbage collector", "un mecanismo que libera automáticamente la memoria que ya no se usa", "a mechanism that automatically frees memory that is no longer used"),
    ("la cola de mensajes", "the message queue", "un búfer que desacopla a los productores de los consumidores de eventos", "a buffer that decouples event producers from consumers"),
    ("la tabla hash", "the hash table", "una estructura que guarda pares clave-valor y busca en tiempo casi constante", "a structure that stores key-value pairs and looks them up in near-constant time"),
    ("el árbol binario de búsqueda", "the binary search tree", "un árbol ordenado donde cada nodo separa los valores menores de los mayores", "an ordered tree where each node separates smaller values from larger ones"),
    ("la red neuronal", "the neural network", "un modelo formado por capas de unidades que aprenden pesos a partir de ejemplos", "a model made of layers of units that learn weights from examples"),
    ("el descenso de gradiente", "gradient descent", "un método que ajusta los parámetros en la dirección que reduce la pérdida", "a method that adjusts parameters in the direction that reduces the loss"),
    ("el sobreajuste", "overfitting", "el fenómeno por el que un modelo memoriza el entrenamiento y generaliza mal", "the phenomenon where a model memorizes the training set and generalizes poorly"),
    ("la validación cruzada", "cross-validation", "una técnica que estima el rendimiento repartiendo los datos en varios pliegues", "a technique that estimates performance by splitting the data into several folds"),
    ("el tokenizador", "the tokenizer", "el componente que convierte el texto en una secuencia de identificadores enteros", "the component that turns text into a sequence of integer identifiers"),
    ("la ventana de contexto", "the context window", "la cantidad máxima de tokens que un modelo puede leer a la vez", "the maximum number of tokens a model can read at once"),
    ("el compilador", "the compiler", "un programa que traduce código fuente a instrucciones de máquina", "a program that translates source code into machine instructions"),
    ("el sistema de archivos", "the file system", "la capa que organiza cómo se guardan y se nombran los datos en disco", "the layer that organizes how data is stored and named on disk"),
    ("la memoria virtual", "virtual memory", "una abstracción que da a cada proceso su propio espacio de direcciones", "an abstraction that gives each process its own address space"),
    ("el protocolo TCP", "the TCP protocol", "un protocolo que entrega bytes de forma ordenada y fiable entre dos máquinas", "a protocol that delivers bytes reliably and in order between two machines"),
    ("el cifrado simétrico", "symmetric encryption", "un esquema que usa la misma clave para cifrar y descifrar", "a scheme that uses the same key to encrypt and decrypt"),
    ("las pruebas unitarias", "unit tests", "comprobaciones automáticas que verifican una función aislada", "automatic checks that verify a single isolated function"),
    ("la integración continua", "continuous integration", "una práctica que compila y prueba cada cambio de forma automática", "a practice that builds and tests every change automatically"),
    ("los microservicios", "microservices", "una arquitectura que divide la aplicación en servicios pequeños e independientes", "an architecture that splits the application into small independent services"),
    ("la paginación", "pagination", "una técnica que devuelve los resultados en bloques de tamaño fijo", "a technique that returns results in fixed-size blocks"),
    ("la limitación de tasa", "rate limiting", "una política que restringe cuántas peticiones puede hacer un cliente por minuto", "a policy that restricts how many requests a client can make per minute"),
    ("la serialización JSON", "JSON serialization", "el proceso de convertir objetos en texto JSON y viceversa", "the process of converting objects to JSON text and back"),
    ("la variable de entorno", "the environment variable", "un par nombre-valor que el sistema operativo entrega a cada proceso", "a name-value pair that the operating system hands to each process"),
    ("el entorno virtual", "the virtual environment", "una carpeta aislada con su propio intérprete de Python y sus paquetes", "an isolated folder with its own Python interpreter and packages"),
    ("el decorador", "the decorator", "una función que envuelve a otra para añadirle comportamiento", "a function that wraps another one to add behavior"),
    ("el generador", "the generator", "una función que produce valores perezosamente con la palabra yield", "a function that lazily produces values with the yield keyword"),
    ("el manejo de excepciones", "exception handling", "el mecanismo para capturar errores sin que el programa se detenga", "the mechanism for catching errors without stopping the program"),
    ("la programación asíncrona", "asynchronous programming", "un modelo que solapa esperas de entrada y salida sin bloquear el hilo", "a model that overlaps input and output waits without blocking the thread"),
    ("el bloqueo de base de datos", "the database lock", "un candado que impide que dos transacciones modifiquen la misma fila a la vez", "a lock that prevents two transactions from modifying the same row at once"),
    ("la migración de esquema", "the schema migration", "un script versionado que cambia la estructura de la base de datos", "a versioned script that changes the structure of the database"),
    ("el registro de eventos", "event logging", "la práctica de escribir mensajes con marca de tiempo sobre lo que hace el sistema", "the practice of writing timestamped messages about what the system does"),
]
HERRAMIENTAS = ["PostgreSQL", "Redis", "nginx", "Docker", "pytest", "Kubernetes", "FastAPI", "SQLite", "Git", "Linux", "Celery", "Prometheus"]
PARAMETROS = ["max_connections", "timeout_seconds", "cache_size", "batch_size", "log_level", "retry_count", "pool_size", "num_workers"]
# aspecto -> (título ES, título EN, palabra clave ES, palabra clave EN, plantillas ES, plantillas EN)
ASPECTOS = {
    "intro": ("{n}: qué es y para qué sirve", "{n}: what it is and what it is for", "qué es", "what is",
        ["Se usa en {t} para simplificar el trabajo diario del equipo.", "Entender {c} es un requisito antes de pasar a producción.", "Su idea central es sencilla y aparece en casi todos los sistemas modernos.", "La documentación de {t} dedica un capítulo completo a {c}."],
        ["It is used in {t} to simplify the daily work of the team.", "Understanding {c} is a prerequisite before going to production.", "Its core idea is simple and appears in almost every modern system.", "The {t} documentation devotes a whole chapter to {c}."]),
    "config": ("{n}: cómo configurarlo", "{n}: how to configure it", "configurar", "configure",
        ["Para configurar {c} edita el archivo de ajustes y reinicia {t}.", "El parámetro {p} controla el comportamiento de {c} y su valor por defecto es {x}.", "Después de cambiar {p} a {x}, comprueba el resultado con un comando de diagnóstico.", "Guarda la configuración de {c} en un archivo versionado y no en la consola."],
        ["To configure {c}, edit the settings file and restart {t}.", "The {p} parameter controls the behavior of {c} and its default value is {x}.", "After changing {p} to {x}, check the result with a diagnostic command.", "Keep the configuration of {c} in a versioned file and not in the console."]),
    "errores": ("{n}: errores frecuentes", "{n}: common errors", "errores", "errors",
        ["Un error frecuente con {c} es olvidar el valor de {p} y dejarlo en {x}.", "Si aparece un fallo intermitente en {t}, revisa primero los registros de {c}.", "Muchos equipos descubren el problema de {c} solo cuando la carga se multiplica por {x}.", "El mensaje de error suele ser engañoso: la causa real está en {c}, no en {t}."],
        ["A common mistake with {c} is forgetting {p} and leaving it at {x}.", "If an intermittent failure shows up in {t}, check the logs of {c} first.", "Many teams only discover the problem with {c} when the load grows by {x} times.", "The error message is usually misleading: the real cause is in {c}, not in {t}."]),
    "rendimiento": ("{n}: rendimiento y optimización", "{n}: performance and optimization", "rendimiento", "performance",
        ["Para medir el rendimiento de {c} usa un perfilador y no tu intuición.", "Subir {p} a {x} mejora la latencia, pero consume más memoria en {t}.", "La optimización prematura de {c} suele complicar el código sin ganar tiempo.", "En una prueba con {x} usuarios, {c} redujo la latencia mediana a la mitad."],
        ["To measure the performance of {c}, use a profiler and not your intuition.", "Raising {p} to {x} improves latency but uses more memory in {t}.", "Premature optimization of {c} usually complicates the code without saving time.", "In a test with {x} users, {c} cut the median latency in half."]),
    "practicas": ("{n}: buenas prácticas", "{n}: best practices", "buenas prácticas", "best practices",
        ["Una buena práctica con {c} es documentar {p} y automatizar su despliegue en {t}.", "Revisa {c} en cada revisión de código y escribe pruebas para los casos límite.", "Mantén {c} simple: añade complejidad solo cuando los datos lo justifiquen.", "Define alertas sobre {c} antes del primer incidente, no después."],
        ["A good practice with {c} is to document {p} and automate its deployment in {t}.", "Review {c} in every code review and write tests for the edge cases.", "Keep {c} simple: add complexity only when the data justifies it.", "Define alerts on {c} before the first incident, not after."]),
    "ejemplo": ("{n}: ejemplo en Python", "{n}: Python example", "ejemplo", "example",
        ["En este ejemplo en Python usamos {t} para mostrar {c} paso a paso.", "El script define {p} = {x} y luego imprime el resultado de {c}.", "Ejecuta el ejemplo con Python 3.11 y compara la salida con la esperada.", "Puedes adaptar el código de {c} a tu propio proyecto cambiando solo {p}."],
        ["In this Python example we use {t} to show {c} step by step.", "The script defines {p} = {x} and then prints the result of {c}.", "Run the example with Python 3.11 and compare the output with the expected one.", "You can adapt the code for {c} to your own project by changing only {p}."]),
    "seguridad": ("{n}: consideraciones de seguridad", "{n}: security considerations", "seguridad", "security",
        ["Desde el punto de vista de la seguridad, {c} nunca debe exponerse sin autenticación.", "Rota las credenciales de {t} cada {x} días y no las guardes en el repositorio.", "Un ataque contra {c} suele aprovechar el valor de {p} mal configurado.", "Audita los accesos a {c} y conserva los registros durante {x} días."],
        ["From a security standpoint, {c} must never be exposed without authentication.", "Rotate the {t} credentials every {x} days and do not store them in the repository.", "An attack on {c} usually exploits a misconfigured {p} value.", "Audit access to {c} and keep the logs for {x} days."]),
}

In [ ]:
# ── Celda 2/3 de carga_corpus · generador sintético ──
# ── Generador del corpus sintético (mismo esquema que el real) ────────────────
import random


def _es_fix(s: str) -> str:
    """Contracciones del español: 'a el' -> 'al', 'de el' -> 'del'."""
    return s.replace(" a el ", " al ").replace(" de el ", " del ").replace("De el ", "Del ").replace("A el ", "Al ")


def _sintetico_wiki(lang: str, n_docs: int, n_queries: int, seed: int = 42):
    """Docs técnicos sintéticos ES/EN: concepto x aspecto x herramienta. Devuelve (docs, queries, qrels)."""
    rng = random.Random(seed + (0 if lang == "es" else 1))
    li = 0 if lang == "es" else 1
    combos = [(c, a, t) for c in range(len(CONCEPTOS)) for a in ASPECTOS for t in range(len(HERRAMIENTAS))]
    rng.shuffle(combos)
    docs, titles = [], []
    for k, (ci, asp, ti) in enumerate(combos[:n_docs]):
        concepto, definicion = CONCEPTOS[ci][li], CONCEPTOS[ci][2 + li]
        tool = HERRAMIENTAS[ti]
        tit_t, plantillas = ASPECTOS[asp][li], ASPECTOS[asp][4 + li]
        nombre = concepto.split(" ", 1)[1] if concepto.split(" ", 1)[0] in ("el", "la", "los", "las", "the") else concepto
        slots = lambda: dict(c=concepto, t=tool, p=rng.choice(PARAMETROS), x=rng.choice([8, 16, 30, 64, 100, 256, 1024]))
        frases = [("{C} " + ("es " if li == 0 else "is ") + definicion + ".").format(C=concepto)]
        frases += [pl.format(**slots()) for pl in rng.sample(plantillas, len(plantillas))]
        frases.append(rng.choice(ASPECTOS["intro"][4 + li]).format(**slots()))
        for _ in range(2):  # ruido: frases de otro concepto (hace el retrieval no trivial)
            otro = rng.randrange(len(CONCEPTOS))
            otra = rng.choice(list(ASPECTOS.values()))[4 + li]
            s = slots(); s["c"] = CONCEPTOS[otro][li]
            frases.append(rng.choice(otra).format(**s))
        s = slots()
        frases.append(("Ejemplo: `{p} = {x}`." if li == 0 else "Example: `{p} = {x}`.").format(**s))
        text = " ".join(f[0].upper() + f[1:] for f in frases)
        docs.append({"id": f"{lang}-syn-{k:04d}", "lang": lang, "text": _es_fix(text) if lang == "es" else text})
        titles.append((nombre, ASPECTOS[asp][2 + li], tool, tit_t.format(n=nombre) + f" ({tool})"))
    queries, qrels = [], {}
    for j, k in enumerate(rng.sample(range(len(docs)), min(n_queries, len(docs)))):
        nombre, kw, tool, titulo = titles[k]
        q = titulo.replace(":", "").replace("(", "").replace(")", "") if j % 2 == 0 else f"{nombre} {kw} {tool}"
        qid = f"{lang}-q{j:04d}"
        queries.append({"qid": qid, "lang": lang, "text": q})
        qrels[qid] = {docs[k]["id"]: 1}
    return docs, queries, qrels

In [ ]:
# ── Celda 3/3 de carga_corpus · cadena oficial → espejo → sintético ──
# ── carga_corpus(): cadena oficial (HF) -> espejo (API de Wikipedia / stdlib) -> sintético ─────
import ast
import json
import os
import random
import urllib.parse
import urllib.request

SEED = 42


def _strip_docstring(func_src: str):
    """Devuelve (código sin docstring, primera frase del docstring) o None si no hay docstring útil."""
    try:
        fn = ast.parse(func_src).body[0]
        doc = ast.get_docstring(fn)
        first = fn.body[0]
    except Exception:
        return None
    if not doc or not (isinstance(first, ast.Expr) and isinstance(getattr(first, "value", None), ast.Constant)):
        return None
    lines = func_src.splitlines()
    code = "\n".join(lines[: first.lineno - 1] + lines[first.end_lineno :])
    frase = doc.strip().split("\n\n")[0].replace("\n", " ").strip()
    frase = frase.split(". ")[0].rstrip(".")
    return (code, frase) if len(frase.split()) >= 4 else None


def _code_stdlib(n_docs: int, n_queries: int, rng: random.Random):
    """Espejo OFFLINE con código real: funciones de la biblioteca estándar con su docstring como query."""
    import importlib, inspect, textwrap
    mods = ["argparse", "bisect", "calendar", "collections", "colorsys", "copy", "csv", "difflib", "fnmatch", "fractions",
            "functools", "getpass", "glob", "heapq", "html", "json", "operator", "os", "pathlib", "posixpath", "queue",
            "random", "re", "shlex", "shutil", "statistics", "string", "tempfile", "textwrap", "urllib.parse", "uuid", "zipfile"]
    cands = []
    for name in mods:
        try:
            src = inspect.getsource(importlib.import_module(name))
            tree = ast.parse(src)
        except Exception:
            continue
        for node in ast.walk(tree):
            if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)) and node.end_lineno - node.lineno >= 4:
                seg = textwrap.dedent("\n".join(src.splitlines()[node.lineno - 1 : node.end_lineno]))
                r = _strip_docstring(seg)
                if r and 200 <= len(r[0]) <= 2500:
                    cands.append((f"{name}.{node.name}", r[0], r[1]))
    rng.shuffle(cands)
    return cands[:n_docs]


def _code_hf(n_docs: int, rng: random.Random):
    """Oficial: CodeSearchNet (Python) en streaming. Prueba varios repos por cambios de formato en `datasets`."""
    from datasets import load_dataset
    errs = []
    for repo, kw in [("code-search-net/code_search_net", {"name": "python"}), ("code_search_net", {"name": "python"}),
                     ("claudios/code_search_net", {"name": "python"}), ("Nan-Do/code-search-net-python", {})]:
        try:
            ds = load_dataset(repo, split="train", streaming=True, **kw).shuffle(seed=SEED, buffer_size=2000)
            out = []
            for ex in ds:
                src = ex.get("func_code_string") or ex.get("whole_func_string") or ex.get("code") or ""
                r = _strip_docstring(src) if '"""' in src or "'''" in src else None
                doc = (ex.get("func_documentation_string") or ex.get("docstring") or "").strip().split("\n\n")[0]
                doc = doc.replace("\n", " ").split(". ")[0].rstrip(".")
                code = r[0] if r else src
                if 200 <= len(code) <= 2500 and len(doc.split()) >= 4:
                    out.append((ex.get("func_name") or f"csn.{len(out)}", code, doc))
                if len(out) >= n_docs:
                    return out
        except Exception as e:  # noqa: BLE001
            errs.append(f"{repo}: {type(e).__name__}")
    raise RuntimeError("CodeSearchNet no disponible -> " + "; ".join(errs))


def _wiki_hf(lang: str, n_docs: int, cap: int):
    """Oficial: wikimedia/wikipedia (snapshot 20231101) en streaming con un buffer pequeño."""
    from datasets import load_dataset
    ds = load_dataset("wikimedia/wikipedia", f"20231101.{lang}", split="train", streaming=True)
    ds = ds.shuffle(seed=SEED, buffer_size=2000)
    out = []
    for ex in ds:
        if len(ex["text"]) >= 600 and len(ex["title"]) >= 3:
            out.append((f"{lang}-{ex['id']}", ex["title"], ex["text"][:cap]))
        if len(out) >= n_docs:
            break
    if len(out) < n_docs // 2:
        raise RuntimeError("pocos artículos")
    return out


def _wiki_api(lang: str, n_docs: int):
    """Espejo: API pública de Wikipedia (artículos aleatorios, solo la sección introductoria; ~20 por petición)."""
    out, seen, tries = [], set(), 0
    while len(out) < n_docs and tries < 80:
        tries += 1
        qs = urllib.parse.urlencode({"action": "query", "format": "json", "generator": "random", "grnnamespace": 0,
                                     "grnlimit": 20, "prop": "extracts", "exintro": 1, "explaintext": 1, "exlimit": 20})
        req = urllib.request.Request(f"https://{lang}.wikipedia.org/w/api.php?{qs}",
                                     headers={"User-Agent": "poliglota-course/1.0 (curso educativo)"})
        with urllib.request.urlopen(req, timeout=20) as r:
            pages = json.load(r).get("query", {}).get("pages", {})
        for p in pages.values():
            if p["pageid"] not in seen and len(p.get("extract", "")) >= 300:
                seen.add(p["pageid"])
                out.append((f"{lang}-{p['pageid']}", p["title"], p["extract"]))
    if len(out) < n_docs // 2:
        raise RuntimeError("pocos artículos")
    return out[:n_docs]


def carga_corpus(fast_dev: bool = True) -> dict:
    """Corpus Políglota -> {"docs": [{id, lang, text}], "queries": [{qid, lang, text}], "qrels": {qid: {doc_id: 1}},
    "source": "real" | "synthetic"}. Cadena: oficial (HF) -> espejo (API Wikipedia; código de la stdlib) -> sintético."""
    rng = random.Random(SEED)
    n_wiki, n_code, q_wiki, q_code, cap = (150, 100, 40, 30, 1500) if fast_dev else (3000, 1500, 200, 150, 6000)
    docs, queries, qrels, source = [], [], {}, "real"
    wiki = {}
    for lang in ("es", "en"):
        for nombre, fn in (("HF wikimedia/wikipedia", lambda: _wiki_hf(lang, n_wiki, cap)),
                           ("API de Wikipedia (espejo)", lambda: _wiki_api(lang, min(n_wiki, 400)))):
            try:
                wiki[lang] = fn()
                print(f"[carga_corpus] Wikipedia {lang}: {len(wiki[lang])} artículos desde {nombre}")
                break
            except Exception as e:  # noqa: BLE001
                print(f"[carga_corpus] {nombre} ({lang}) falló: {type(e).__name__}: {str(e)[:80]}")
    if len(wiki) == 2:
        for lang, arts in wiki.items():
            docs += [{"id": i, "lang": lang, "text": t} for i, _, t in arts]
            for j, k in enumerate(rng.sample(range(len(arts)), min(q_wiki, len(arts)))):
                qid = f"{lang}-q{j:04d}"
                queries.append({"qid": qid, "lang": lang, "text": arts[k][1]})
                qrels[qid] = {arts[k][0]: 1}
    else:
        source = "synthetic"
        print("⚠️  [carga_corpus] SIN acceso a Wikipedia: uso texto ES/EN SINTÉTICO (mismo esquema). Los números NO son comparables con datos reales.")
        for lang in ("es", "en"):
            d, q, r = _sintetico_wiki(lang, n_wiki, q_wiki, SEED)
            docs += d; queries += q; qrels.update(r)
    try:
        code = _code_hf(n_code, rng)
        print(f"[carga_corpus] código: {len(code)} funciones desde CodeSearchNet (HF)")
    except Exception as e:  # noqa: BLE001
        print(f"[carga_corpus] CodeSearchNet falló ({str(e)[:90]}); uso funciones REALES de la stdlib de Python (espejo offline)")
        code = _code_stdlib(n_code, q_code, rng)
    for k, (name, src, doc) in enumerate(code):
        docs.append({"id": f"code-{k:04d}", "lang": "code", "text": src})
    for j, k in enumerate(rng.sample(range(len(code)), min(q_code, len(code)))):
        qid = f"code-q{j:04d}"
        queries.append({"qid": qid, "lang": "code", "text": code[k][2]})
        qrels[qid] = {f"code-{k:04d}": 1}
    return {"docs": docs, "queries": queries, "qrels": qrels, "source": source}

In [ ]:
t0 = time.time()
corpus = carga_corpus(fast_dev=FAST_DEV_RUN)
docs, queries, qrels, source = corpus["docs"], corpus["queries"], corpus["qrels"], corpus["source"]
print(f"\nsource = {source!r} | {len(docs)} docs | {len(queries)} queries | {time.time() - t0:.1f} s")
for lang in ("es", "en", "code"):
    d = next(x for x in docs if x["lang"] == lang); q = next(x for x in queries if x["lang"] == lang)
    print(f"\n[{lang}] doc {d['id']}: {d['text'][:140]!r}...\n[{lang}] query {q['qid']}: {q['text']!r} → relevante {list(qrels[q['qid']])}")

In [ ]:
import pandas as pd
df = pd.DataFrame(docs)
df["chars"] = df.text.str.len(); df["bytes"] = df.text.map(lambda t: len(t.encode("utf-8"))); df["palabras"] = df.text.str.split().map(len)

fig, axs = plt.subplots(1, 2, figsize=(11, 3.8))
for lang, c in zip(("es", "en", "code"), (AZUL, NARANJA, VERDE)):
    axs[0].hist(df[df.lang == lang].chars, bins=30, alpha=0.6, color=c, label=lang)
axs[0].set_xlabel("caracteres por documento"); axs[0].set_ylabel("nº de documentos"); axs[0].set_title("Longitud de los documentos"); axs[0].legend()
g = df.groupby("lang")[["chars", "bytes"]].sum()
axs[1].bar(g.index, (g.bytes / g.chars - 1) * 100, color=[VERDE, NARANJA, AZUL])  # índice ordenado: code, en, es
axs[1].set_ylabel("bytes extra por cada 100 caracteres"); axs[1].set_title("Sobrecoste de bytes sobre ASCII puro")
plt.tight_layout(); plt.show()
display(df.groupby("lang")[["chars", "bytes", "palabras"]].mean().round(1))

> 👀 **Qué debes observar.** 1. **Bytes extra**: el inglés y el código rondan 0 (ASCII puro: 1 byte por carácter); el español queda por encima por las vocales acentuadas y la «ñ» (2 bytes cada una). Con el corpus sintético el efecto es pequeño; con Wikipedia real mide tú el valor. Es una pista del costo oculto de los idiomas con diacríticos (módulo 01 lo cuantifica con 12 idiomas).
> 2. Los documentos de código y los de Wikipedia tienen distribuciones de longitud distintas: cualquier hiperparámetro que dependa de la longitud (como $b$ en BM25) se comportará distinto por tipo de corpus.

## 8. Métricas de retrieval canónicas

Definimos una sola vez (y se copian idénticas en todos los módulos) tres métricas. Sea $\text{rel}_i$ la relevancia del documento en la posición $i$ del ranking y $k$ el corte:

$$\text{DCG@}k=\sum_{i=1}^{k}\frac{\text{rel}_i}{\log_2(i+1)},\qquad \text{nDCG@}k=\frac{\text{DCG@}k}{\text{IDCG@}k}$$

* $\text{IDCG@}k$ es el DCG del **ranking ideal** (relevantes ordenados por relevancia decreciente): normaliza a $[0,1]$.
* $\dfrac{1}{\log_2(i+1)}$ es el **descuento por posición**: acertar en el puesto 1 vale 1.0; en el 2, 0.63; en el 10, 0.29.
* $\text{Recall@}k=\dfrac{|\{\text{relevantes}\}\cap\{\text{top-}k\}|}{|\{\text{relevantes}\}|}$.
* $\text{MRR@}k=\dfrac{1}{\text{rango del primer relevante}}$ si está en el top-$k$; si no, 0 (promediado sobre consultas).

**Ejemplo resuelto a mano.** Ranking `[d3, d1, d7, d2]`, relevantes `{d1, d2}` (relevancia 1):

* DCG@4 = $\dfrac{1}{\log_2 3}+\dfrac{1}{\log_2 5}=0.6309+0.4307=1.0616$ (d1 en la posición 2, d2 en la 4).
* IDCG@4 = $\dfrac{1}{\log_2 2}+\dfrac{1}{\log_2 3}=1+0.6309=1.6309$.
* nDCG@4 = $1.0616/1.6309=\mathbf{0.651}$. Recall@4 = $2/2=1$. MRR@4 = $1/2=0.5$.

**Convenciones (importantes y a menudo mal implementadas):** ganancia **lineal** (la de `trec_eval`/`pytrec_eval`, que usan BEIR y MTEB); **duplicados** en el ranking cuentan una sola vez; las consultas **sin relevantes** devuelven 0 y `evaluate()` las **excluye** del promedio; los **empates** de score se rompen por orden del corpus (determinista).

In [ ]:
# ── Métricas canónicas de retrieval (idénticas en todos los módulos) ──
# ── Métricas de retrieval canónicas de Políglota (módulo 00) ──────────────────
import math

import numpy as np


def _dedup(ranked_ids):
    """Quita duplicados conservando la primera aparición (un doc repetido NO puntúa dos veces)."""
    seen, out = set(), []
    for d in ranked_ids:
        if d not in seen:
            seen.add(d)
            out.append(d)
    return out


def dcg_at_k(ranked_ids, rels: dict, k: int = 10) -> float:
    """DCG con ganancia lineal (como trec_eval / pytrec_eval, que usa BEIR/MTEB) y descuento 1/log2(rango+1)."""
    ranked = _dedup(ranked_ids)[:k]
    return sum(rels.get(d, 0) / math.log2(i + 2) for i, d in enumerate(ranked) if rels.get(d, 0) > 0)


def ndcg_at_k(ranked_ids, rels: dict, k: int = 10) -> float:
    """nDCG@k. rels: doc_id -> relevancia (>0 relevante). Sin relevantes -> 0.0 (evaluate() las excluye)."""
    ideal = sorted((r for r in rels.values() if r > 0), reverse=True)[:k]
    idcg = sum(r / math.log2(i + 2) for i, r in enumerate(ideal))
    return 0.0 if idcg == 0 else dcg_at_k(ranked_ids, rels, k) / idcg


def recall_at_k(ranked_ids, rels: dict, k: int = 10) -> float:
    """Fracción de los documentos relevantes que aparecen en el top-k."""
    relevant = {d for d, r in rels.items() if r > 0}
    if not relevant:
        return 0.0
    return len(relevant & set(_dedup(ranked_ids)[:k])) / len(relevant)


def mrr_at_k(ranked_ids, rels: dict, k: int = 10) -> float:
    """1/rango del primer relevante dentro del top-k (0 si no hay ninguno)."""
    for i, d in enumerate(_dedup(ranked_ids)[:k]):
        if rels.get(d, 0) > 0:
            return 1.0 / (i + 1)
    return 0.0


def rank_from_scores(scores, doc_ids, k: int | None = None):
    """Ordena por score desc. Empates -> orden original del corpus (determinista). Los empates son una fuente
    silenciosa de varianza: con scores iguales, el orden de entrada decidiría el resultado."""
    scores = np.asarray(scores, dtype=float)
    order = np.lexsort((np.arange(len(scores)), -scores))
    order = order[:k] if k else order
    return [doc_ids[i] for i in order]


def evaluate(run: dict, qrels: dict, k: int = 10) -> dict:
    """run: {qid: [doc_id ordenados]}. Promedia solo sobre queries con algún relevante (convención trec_eval)."""
    n, tot = 0, {"ndcg": 0.0, "recall": 0.0, "mrr": 0.0}
    for qid, rels in qrels.items():
        if not any(r > 0 for r in rels.values()):
            continue
        ranked = run.get(qid, [])
        tot["ndcg"] += ndcg_at_k(ranked, rels, k)
        tot["recall"] += recall_at_k(ranked, rels, k)
        tot["mrr"] += mrr_at_k(ranked, rels, k)
        n += 1
    out = {m: (v / n if n else 0.0) for m, v in tot.items()}
    out.update(n_queries=n, n_skipped=len(qrels) - n)
    return out

In [ ]:
# Tests: el ejemplo a mano + casos límite
r = ["d3", "d1", "d7", "d2"]; rel = {"d1": 1, "d2": 1}
assert abs(ndcg_at_k(r, rel, 4) - 0.6510) < 1e-3
assert recall_at_k(r, rel, 4) == 1.0 and mrr_at_k(r, rel, 4) == 0.5
assert ndcg_at_k(["d1", "d1", "d1"], {"d1": 1}, 10) == 1.0            # duplicados no inflan el DCG
assert ndcg_at_k(r, {}, 10) == 0.0 and recall_at_k(r, {}, 10) == 0.0   # sin relevantes: 0 (evaluate() las excluye)
assert rank_from_scores([0.5, 0.9, 0.9], ["a", "b", "c"]) == ["b", "c", "a"]  # empate b/c -> orden del corpus
graded = {"a": 3, "b": 2, "c": 0}                                     # relevancias graduadas
print("nDCG@3 graduado, orden ideal:", ndcg_at_k(["a", "b", "c"], graded, 3), "| invertido:", round(ndcg_at_k(["b", "a", "c"], graded, 3), 3))
print("✔ métricas OK")

In [ ]:
i = np.arange(1, 11)
fig, ax = plt.subplots(figsize=(6.5, 3.6))
ax.plot(i, 1 / np.log2(i + 1), "o-", color=AZUL, label=r"$1/\log_2(i+1)$ (nDCG)")
ax.plot(i, 1 / i, "s--", color=NARANJA, label=r"$1/i$ (MRR)")
ax.set_xlabel("posición en el ranking $i$"); ax.set_ylabel("peso de un acierto"); ax.set_title("¿Cuánto vale acertar en la posición i?")
ax.set_xticks(i); ax.legend(); plt.tight_layout(); plt.show()

> 👀 **Qué debes observar.** El descuento de nDCG es **más suave** que el de MRR: en la posición 10 un acierto aún vale 0.29 de uno en la 1ª posición (con MRR vale 0.10). Por eso MRR castiga mucho más no poner el relevante primero, y nDCG@10 tolera listas con varios relevantes. Elegir la métrica es elegir qué comportamiento premias.

## 9. Baseline léxico: BM25 desde cero

### Intuición
BM25 puntúa un documento $D$ para una consulta $Q$ sumando, por cada término de la consulta, **cuánta evidencia** aporta: (a) cuán *raro* es el término en el corpus (IDF), (b) cuántas veces aparece en el documento con **saturación** (la 10.ª aparición aporta mucho menos que la 2.ª) y (c) una **normalización por longitud** (un documento largo menciona más términos «por azar»).

### Fórmula
$$\text{BM25}(Q,D)=\sum_{t\in Q}\text{IDF}(t)\cdot\frac{f(t,D)\,(k_1+1)}{f(t,D)+k_1\left(1-b+b\,\dfrac{|D|}{\text{avgdl}}\right)},\qquad \text{IDF}(t)=\ln\!\left(1+\frac{N-n_t+0.5}{n_t+0.5}\right)$$

* $f(t,D)$: veces que el término $t$ aparece en $D$ (*term frequency*). $|D|$: longitud de $D$ en términos; $\text{avgdl}$: longitud media del corpus.
* $N$: nº de documentos; $n_t$: nº de documentos que contienen $t$ (*document frequency*).
* $k_1\approx1.2$–$2$ controla la **saturación** de $f$; $b\in[0,1]$ controla cuánto se **normaliza por longitud** ($b=0$: nada; $b=1$: total).
* Esta variante de IDF (la de Lucene) nunca es negativa; la de `rank_bm25.BM25Okapi` usa $\ln\frac{N-n_t+0.5}{n_t+0.5}$ y **sí puede serlo** para términos en más de la mitad de los documentos (la librería los sustituye por un suelo $\epsilon\cdot\overline{\text{IDF}}$).

**Ejemplo resuelto a mano** ($k_1=1.2$, $b=0.75$). Corpus: D1 = «gato negro», D2 = «gato gato blanco», D3 = «perro negro»; consulta «gato».

* $N=3$, $n_{\text{gato}}=2$ ⇒ $\text{IDF}=\ln(1+1.5/2.5)=\ln 1.6=0.470$. $\text{avgdl}=(2+3+2)/3=2.33$.
* D1 ($f=1$, $|D|=2$): $0.470\cdot\dfrac{1\cdot2.2}{1+1.2\,(0.25+0.75\cdot 2/2.33)}=0.470\cdot\dfrac{2.2}{2.071}=\mathbf{0.499}$.
* D2 ($f=2$, $|D|=3$): $0.470\cdot\dfrac{2\cdot2.2}{2+1.2\,(0.25+0.75\cdot 3/2.33)}=0.470\cdot\dfrac{4.4}{3.457}=\mathbf{0.598}$.
* D3 ($f=0$): 0. Ranking: D2 > D1 > D3. Fíjate en que D2 tiene **el doble** de apariciones pero solo un **20 %** más de score: saturación + penalización por longitud. La celda que sigue a la implementación lo comprueba con el `BM25` propio.

**Implementación:** un **índice invertido** (`término → [(doc, tf)]`) que solo visita los documentos que contienen algún término de la consulta. Es la misma estructura que usa Lucene/Elasticsearch (analogía directa con la búsqueda de candidatos de un recsys).

In [ ]:
ks = np.linspace(0, 20, 200)
fig, axs = plt.subplots(1, 2, figsize=(11, 3.8))
for k1, c in zip((0.5, 1.2, 2.0, 5.0), (AZUL, NARANJA, VERDE, GRIS)):
    axs[0].plot(ks, ks * (k1 + 1) / (ks + k1), color=c, label=f"$k_1={k1}$")
axs[0].plot(ks, ks, "k:", lw=1, label="sin saturación (lineal)")
axs[0].set_xlabel("frecuencia del término $f(t,D)$"); axs[0].set_ylabel("contribución (sin IDF)"); axs[0].set_title("Saturación de la frecuencia"); axs[0].legend(); axs[0].set_ylim(0, 12)
for b, c in zip((0.0, 0.5, 0.75, 1.0), (GRIS, VERDE, AZUL, NARANJA)):
    ratio = np.linspace(0.2, 4, 100)   # |D| / avgdl
    axs[1].plot(ratio, 3 * 2.2 / (3 + 1.2 * (1 - b + b * ratio)), color=c, label=f"$b={b}$")
axs[1].set_xlabel("longitud relativa $|D|/\\mathrm{avgdl}$"); axs[1].set_ylabel("contribución con $f=3$, $k_1=1.2$"); axs[1].set_title("Normalización por longitud"); axs[1].legend()
plt.tight_layout(); plt.show()

> 👀 **Qué debes observar.** **Izquierda.** Con $k_1$ pequeño la curva se aplana enseguida (la 3.ª aparición casi no suma); con $k_1$ grande se acerca al conteo lineal. **Derecha.** Con $b=0$ los documentos largos no se penalizan (curva plana); con $b=1$ la contribución cae a la mitad cuando el documento es el doble de largo que el promedio. Estos dos botones son los únicos hiperparámetros relevantes de BM25.

In [ ]:
# ── Tokenización léxica simple + BM25 propio con índice invertido ──
# ── Tokenización simple + BM25 propio (módulo 00) ─────────────────────────────
import math
import re
import unicodedata
from collections import Counter, defaultdict

import numpy as np

_WORD_RE = re.compile(r"\w+", re.UNICODE)


def fold_accents(text: str) -> str:
    """'canción' -> 'cancion' (NFD y descartar marcas combinantes Mn)."""
    nfd = unicodedata.normalize("NFD", text)
    return unicodedata.normalize("NFC", "".join(c for c in nfd if unicodedata.category(c) != "Mn"))


def tokenize_simple(text: str, accents: str = "keep") -> list[str]:
    """Minúsculas + palabras \\w+. accents='fold' quita tildes. (Esto NO es el 'token' de un LLM: es un token léxico.)"""
    text = unicodedata.normalize("NFC", text).lower()
    if accents == "fold":
        text = fold_accents(text)
    return _WORD_RE.findall(text)


class BM25:
    """BM25 con índice invertido. idf='lucene': ln(1+(N-n+.5)/(n+.5)) (siempre >= 0).
    idf='okapi': ln((N-n+.5)/(n+.5)) con suelo epsilon*idf_medio (como rank_bm25.BM25Okapi)."""

    def __init__(self, k1: float = 1.2, b: float = 0.75, idf: str = "lucene", epsilon: float = 0.25):
        self.k1, self.b, self.idf_kind, self.epsilon = k1, b, idf, epsilon

    def fit(self, docs_tokens: list[list[str]], doc_ids: list[str]) -> "BM25":
        self.doc_ids = list(doc_ids)
        self.N = len(docs_tokens)
        self.dl = np.array([len(t) for t in docs_tokens], dtype=float)
        self.avgdl = float(self.dl.mean()) if self.N else 0.0
        self.postings = defaultdict(list)  # término -> [(índice_doc, tf)]
        for i, toks in enumerate(docs_tokens):
            for term, tf in Counter(toks).items():
                self.postings[term].append((i, tf))
        raw = {t: len(p) for t, p in self.postings.items()}
        if self.idf_kind == "lucene":
            self.idf = {t: math.log(1 + (self.N - n + 0.5) / (n + 0.5)) for t, n in raw.items()}
        else:
            self.idf = {t: math.log(self.N - n + 0.5) - math.log(n + 0.5) for t, n in raw.items()}
            floor = self.epsilon * sum(self.idf.values()) / len(self.idf)
            self.idf = {t: (v if v >= 0 else floor) for t, v in self.idf.items()}
        return self

    def scores(self, q_tokens: list[str]) -> np.ndarray:
        s = np.zeros(self.N)
        for term in q_tokens:  # los términos repetidos de la query suman varias veces (igual que rank_bm25)
            for i, tf in self.postings.get(term, ()):
                norm = self.k1 * (1 - self.b + self.b * self.dl[i] / self.avgdl)
                s[i] += self.idf[term] * tf * (self.k1 + 1) / (tf + norm)
        return s

    def search(self, q_tokens: list[str], k: int = 10) -> list[str]:
        s = self.scores(q_tokens)
        order = np.lexsort((np.arange(self.N), -s))[:k]  # empates -> orden del corpus
        return [self.doc_ids[i] for i in order]

In [ ]:
# Comprobación del ejemplo resuelto a mano (D1, D2, D3 y la consulta «gato»)
mini = BM25(k1=1.2, b=0.75).fit([["gato", "negro"], ["gato", "gato", "blanco"], ["perro", "negro"]], ["D1", "D2", "D3"])
print(np.round(mini.scores(["gato"]), 3), mini.search(["gato"], 3))   # esperado: [0.499 0.598 0.] ['D2', 'D1', ...]

In [ ]:
def construir_bm25(docs, acentos="keep", **kw):
    bm = BM25(**kw).fit([tokenize_simple(d["text"], acentos) for d in docs], [d["id"] for d in docs])
    return bm

def correr(bm, queries, acentos="keep", k=10, tok=tokenize_simple):
    return {q["qid"]: bm.search(tok(q["text"], acentos), k) for q in queries}

t0 = time.time()
bm25 = construir_bm25(docs)
run = correr(bm25, queries)
print(f"indexado + {len(queries)} consultas en {time.time() - t0:.2f} s | vocabulario léxico: {len(bm25.postings):,} términos")

def por_idioma(run, qrels, k=10):
    filas = []
    for lang in ("es", "en", "code", None):
        qr = {q["qid"]: qrels[q["qid"]] for q in queries if lang is None or q["lang"] == lang}
        filas.append({"grupo": lang or "TODAS", **evaluate(run, qr, k)})
    return pd.DataFrame(filas).set_index("grupo")

res = por_idioma(run, qrels)
display(res.round(3))

In [ ]:
# Validación cruzada con rank_bm25 (misma fórmula de IDF y mismos k1, b)
from rank_bm25 import BM25Okapi
toks = [tokenize_simple(d["text"]) for d in docs]
ref = BM25Okapi(toks, k1=1.5, b=0.75)
mio = BM25(k1=1.5, b=0.75, idf="okapi").fit(toks, [d["id"] for d in docs])
q = tokenize_simple(queries[0]["text"])
diff = np.abs(ref.get_scores(q) - mio.scores(q)).max()
print(f"máx |score_rank_bm25 − score_propio| = {diff:.2e}")
assert diff < 1e-9
run_ref = {x["qid"]: rank_from_scores(ref.get_scores(tokenize_simple(x["text"])), [d["id"] for d in docs], 10) for x in queries}
run_mio = correr(mio, queries)
print("nDCG@10 rank_bm25:", round(evaluate(run_ref, qrels)["ndcg"], 4), "| propio (idf=okapi):", round(evaluate(run_mio, qrels)["ndcg"], 4),
      "| propio (idf=lucene, k1=1.2):", round(res.loc["TODAS", "ndcg"], 4))

In [ ]:
ax = res.loc[["es", "en", "code", "TODAS"], ["ndcg", "recall", "mrr"]].plot.bar(figsize=(8, 3.8), color=[AZUL, NARANJA, VERDE], rot=0, width=0.8)
ax.set_ylim(0, 1.05); ax.set_ylabel("valor de la métrica @10"); ax.set_title(f"Baseline BM25 por tipo de consulta (corpus {source})")
for c in ax.containers: ax.bar_label(c, fmt="%.2f", fontsize=8)
plt.tight_layout(); plt.show()

> 👀 **Qué debes observar.** 1. El BM25 propio y `rank_bm25` coinciden hasta el error numérico cuando usan la misma fórmula de IDF: ya sabes **qué** hace la librería.
> 2. Compara las barras: **código** puntúa claramente por debajo de ES/EN. Las consultas de código son frases en inglés (docstrings) y los documentos son identificadores como `join_path`: poco solapamiento de *cadenas*. Es un problema de **vocabulario** (y de tokenización), no de BM25.
> 3. Si `source == "synthetic"`, ES y EN salen casi perfectos porque el generador es repetitivo: no extraigas conclusiones sobre Wikipedia.

## 10. Experimentos: dónde falla un buscador léxico

### 10.1 La barrera del idioma
BM25 compara **cadenas**. Si la consulta está en español y el documento relevante en inglés (o al revés), solo se solapan cognados y nombres propios. Probemos con 6 pares de oraciones paralelas escritas a mano.

In [ ]:
pares = [("El gato duerme en el sofá", "The cat sleeps on the sofa"),
         ("Configura la caché del servidor", "Configure the server cache"),
         ("La base de datos está bloqueada", "The database is locked"),
         ("Instala el paquete con pip", "Install the package with pip"),
         ("El balanceador reparte las peticiones", "The load balancer spreads the requests"),
         ("Reinicia el contenedor de Docker", "Restart the Docker container")]
es_docs = [a for a, _ in pares]; en_docs = [b for _, b in pares]
def acierto(consultas, coleccion):
    bm = BM25().fit([tokenize_simple(t) for t in coleccion], [str(i) for i in range(len(coleccion))])
    top1 = [bm.search(tokenize_simple(c), 1)[0] for c in consultas]
    sin_solape = [bm.scores(tokenize_simple(c)).max() == 0 for c in consultas]
    # un «acierto» con score 0 sería un empate resuelto por el orden del corpus (suerte), no recuperación
    return np.mean([int(t) == i and not z for i, (t, z) in enumerate(zip(top1, sin_solape))]), sum(sin_solape)
print("ES → ES (mismo idioma):  acierto@1 = %.2f | consultas sin NINGÚN término en común: %d" % acierto(es_docs, es_docs))
print("ES → EN (traducción):    acierto@1 = %.2f | consultas sin NINGÚN término en común: %d" % acierto(es_docs, en_docs))

> 👀 **Qué debes observar.** Con el mismo idioma el acierto es 1.0; cruzando idiomas la mayoría de las consultas **no comparte ni un término** con su traducción (los pocos aciertos se deben a cognados y nombres propios como «Docker», «pip» o «cache/caché» tras plegar mayúsculas). Fíjate además en que no contamos como acierto un top-1 con score 0: sería un empate resuelto por el orden del corpus. Un embedder multilingüe resolverá esto (módulos 10–12); mientras tanto, es la razón de ser de Políglota.

### 10.2 La tokenización importa incluso para BM25
En código, las consultas dicen «join path components» y los documentos contienen `os.path.join(a, *p)`. Una tokenización que **divide identificadores** (`snake_case`, `camelCase`) hace visibles las palabras. Medimos el efecto sobre las consultas de código.

In [ ]:
import re
def split_ident(text, acentos="keep"):
    """Divide snake_case y camelCase además de \\w+: 'getHTTPResponse_code' -> ['get','http','response','code']."""
    enteros = [w for w in tokenize_simple(text, acentos) if "_" in w]   # conserva también el identificador entero
    texto = re.sub(r"([a-z0-9])([A-Z])", r"\1 \2", text)          # camelCase
    texto = re.sub(r"([A-Z]+)([A-Z][a-z])", r"\1 \2", texto)       # HTTPResponse
    texto = texto.replace("_", " ")
    return tokenize_simple(texto, acentos) + enteros

code_docs = [d for d in docs if d["lang"] == "code"]; code_q = [q for q in queries if q["lang"] == "code"]
qr_code = {q["qid"]: qrels[q["qid"]] for q in code_q}
resultados = {}
for nombre, tok in (("\\w+ (palabras)", tokenize_simple), ("dividir identificadores", split_ident)):
    bm = BM25().fit([tok(d["text"]) for d in code_docs], [d["id"] for d in code_docs])
    run_c = {q["qid"]: bm.search(tok(q["text"]), 10) for q in code_q}
    resultados[nombre] = evaluate(run_c, qr_code)["ndcg"]
print({k: round(v, 3) for k, v in resultados.items()})
fig, ax = plt.subplots(figsize=(5.5, 3.2))
ax.bar(list(resultados), list(resultados.values()), color=[GRIS, VERDE]); ax.set_ylabel("nDCG@10 (consultas de código)")
ax.set_title("Efecto de la tokenización léxica en código"); plt.tight_layout(); plt.show()

> 👀 **Qué debes observar.** Dividir identificadores es una decisión de **tokenización** (no de modelo) y cambia el nDCG@10 de código. Esto anticipa el tema del curso: *cómo cortes el texto determina qué puede encontrar el sistema*. Si la mejora te parece pequeña o grande, recuerda que depende del corpus (real/sintético) y del número de consultas: la siguiente sección cuantifica esa incertidumbre.

### 10.3 ¿Cuánta incertidumbre tiene tu nDCG? Intervalo bootstrap
Con unas decenas de consultas, el nDCG@10 medio es una estimación ruidosa. Remuestreamos las **consultas** con reemplazo (*bootstrap*) para obtener un intervalo del 95 %.

In [ ]:
def ndcg_por_query(run, qrels, k=10):
    return np.array([ndcg_at_k(run[qid], rels, k) for qid, rels in qrels.items()])

def bootstrap_ci(valores, n_boot=2000, alpha=0.05, seed=SEED):
    rng = np.random.default_rng(seed)
    medias = rng.choice(valores, size=(n_boot, len(valores)), replace=True).mean(axis=1)
    return valores.mean(), np.quantile(medias, [alpha / 2, 1 - alpha / 2]), medias

v = ndcg_por_query(run, qrels)
media, (lo, hi), medias = bootstrap_ci(v)
print(f"nDCG@10 = {media:.3f}  IC95% [{lo:.3f}, {hi:.3f}]  (n = {len(v)} consultas)")
fig, ax = plt.subplots(figsize=(6.5, 3.4))
ax.hist(medias, bins=40, color=AZUL, alpha=0.8); ax.axvline(media, color=NARANJA, lw=2, label="media")
ax.axvline(lo, color=GRIS, ls="--", label="IC 95 %"); ax.axvline(hi, color=GRIS, ls="--")
ax.set_xlabel("nDCG@10 medio de cada remuestreo"); ax.set_ylabel("frecuencia"); ax.set_title("Bootstrap sobre las consultas"); ax.legend()
plt.tight_layout(); plt.show()

> 👀 **Qué debes observar.** El intervalo tiene una anchura de varios puntos de nDCG. **Si dos sistemas difieren menos que eso, no puedes afirmar que uno sea mejor** con este golden set. El módulo 15 formaliza esto (tests pareados, tamaño de muestra); aquí ya tienes el hábito: *reporta siempre un intervalo*.

## 11. Mapa de madurez
Estado a **octubre de 2026**. 🟢 sólido (estándar de industria) · 🟡 emergente (resultados publicados, adopción parcial) · 🔴 experimental.


| Técnica | Madurez (oct-2026) | Quién la usa (respaldado) | Fuente |
|---|---|---|---|
| BM25 / retrieval léxico con índice invertido | 🟢 | Lucene y los motores construidos sobre él; baseline de BEIR | Thakur et al. 2021 (BEIR, arXiv:2104.08663); Robertson & Zaragoza 2009 |
| Byte-level BPE (tokenización de LLMs) | 🟢 | GPT-2 y la familia de encodings de `tiktoken` (modelo → encoding en `tiktoken/model.py`) | Radford et al. 2019; Wang et al. 2019 (arXiv:1909.03341); repositorio `tiktoken` |
| Embeddings de oraciones con bi-encoders | 🟢 | Sentence-BERT y sucesores | Reimers & Gurevych 2019 (arXiv:1908.10084) |
| Retrieval híbrido (embeddings + BM25) | 🟢 | Anthropic documenta «Contextual Retrieval» combinando ambos (sep-2024) | anthropic.com/news/contextual-retrieval |
| Sparse aprendido (SPLADE) y late interaction (ColBERT) | 🟡 | Resultados fuertes en BEIR; adopción parcial en producción | Formal et al. 2021 (arXiv:2107.05720); Khattab & Zaharia 2020 (arXiv:2004.12832) |
| Modelos sin tokenizer (p. ej. Byte Latent Transformer) | 🔴 | Investigación de Meta FAIR; no es el estándar de producción | Pagnoni et al. 2024 (arXiv:2412.09871) |

## 12. 🏭 En producción
* **BM25 sigue vivo.** En sistemas reales casi nunca se sustituye el retrieval léxico: se **combina** con embeddings (*hybrid search*). Anthropic describe en su post «Introducing Contextual Retrieval» (sep-2024) que combinar embeddings con BM25 reduce los fallos de recuperación frente a usar solo embeddings (sus cifras son de su propio benchmark; no las generalices).
* **Tokenización = costo.** Las APIs de LLM cobran por token; el impuesto de tokens entre idiomas (Ahia et al., 2023; Petrov et al., 2023) es un factor de costo y de equidad que se decide *antes* del modelo.
* **El pipeline se versiona entero.** Cambiar el tokenizer o la normalización invalida los índices (hay que re-indexar): es una decisión de ingeniería, no un hiperparámetro.
* **Coste de este módulo:** CPU, segundos. Un BM25 sobre millones de documentos se hace con Lucene/Elasticsearch/OpenSearch; aquí lo implementamos para entender qué hacen.

**🧪 ¿Cuánto cuesta de verdad una prima de tokens?** Si tu idioma necesita $r$ veces más tokens para el mismo texto, el coste **no** siempre es $r$. Con la aproximación de Kaplan et al. (2020) para el cómputo del *forward* por token, $C_{\text{fwd}}\approx 2N + 2\,n_{\text{layer}}\,n_{\text{ctx}}\,d_{\text{model}}$ ($N$ = parámetros sin embeddings), procesar $n$ tokens cuesta $\approx n\,(2N + 2\,n_{\text{layer}}\,n\,d_{\text{model}})$ (cota con el contexto completo): el término lineal crece como $r$ y el de atención como $r^2$. Además, en una ventana fija cabe $1/r$ del texto. La celda lo calcula para una configuración de tamaño 7 B (32 capas, $d=4096$; números ilustrativos).

In [ ]:
def flops_forward(n_tok: int, N: float = 6.5e9, L: int = 32, d: int = 4096) -> float:
    """Kaplan et al. (2020): C_fwd por token ≈ 2N + 2·L·n_ctx·d; aquí n_ctx = n_tok (cota superior)."""
    return n_tok * (2 * N + 2 * L * n_tok * d)

for n_base in (1_000, 32_000):
    for r in (1.5, 3.0):
        coste = flops_forward(int(r * n_base)) / flops_forward(n_base)
        print(f"texto que en inglés ocupa {n_base:>6,} tokens | prima r={r}: x{r:.1f} en factura por token, x{coste:.2f} en FLOPs, {1/r:.0%} del texto en la misma ventana")

> 👀 **Qué debes observar.** Con textos cortos el sobrecoste en cómputo ≈ $r$ (domina $2N$); con contextos largos supera claramente a $r$ (tiende a $r^2$ cuanto más domina la atención; con 32 k tokens y $r=3$ ya es ≈ 5×). Por eso el «impuesto de tokens» de un idioma es un problema de **latencia y de capacidad de servicio**, no solo de factura, y por eso los laboratorios lo miden antes de fijar el tokenizer (módulo 04).

## 13. 🧠 Secretos de la élite
1. **Empieza SIEMPRE por un baseline léxico.** BEIR mostró que BM25 es un baseline robusto frente a modelos neuronales en evaluación *zero-shot* (Thakur et al., 2021). Si tu embedder no gana a BM25 en tu dominio, el problema no es el modelo.
2. **Los empates cambian tus resultados.** Con scores iguales, el orden de entrada decide el ranking: `rank_from_scores` rompe los empates por orden de corpus para que el experimento sea reproducible. Es un fallo de evaluación clásico (lo verificaste en el test de arriba).
3. **Duplicados y consultas sin relevantes.** Un doc repetido en el ranking inflaría el DCG si no se deduplica; una consulta sin relevantes dividiría por cero. Toda implementación seria fija una convención y la documenta (aquí: dedup + exclusión).
4. **El tamaño del subconjunto cambia el nivel absoluto.** Con un corpus más grande hay más distractores: el nDCG baja. Compara sistemas **solo sobre el mismo corpus y las mismas consultas** (corre `FAST_DEV_RUN = False` y compruébalo).
5. **Golden sets de «ítem conocido» están sesgados.** Si la consulta es el título del documento, los modelos léxicos salen favorecidos (comparten palabras con el texto). Para comparar embedders hay que construir consultas que **no** copien el vocabulario del documento (módulo 15).
6. **El IDF de la librería no es el de Lucene.** `rank_bm25.BM25Okapi` puede dar IDF negativo y lo corrige con un suelo; Lucene usa $\ln(1+\cdot)$. Dos «BM25» distintos dan rankings distintos: documenta cuál usas.
7. **Reporta intervalos de confianza.** Con ≈ 100 consultas, ±3–5 puntos de nDCG son ruido (mídelo con el bootstrap de arriba, que usa tus datos).
8. **La tokenización léxica es una variable experimental.** Quitar tildes, dividir identificadores o usar stemming mueve el nDCG tanto como cambiar de algoritmo de ranking; anótala en cada experimento.

## 14. ⚠️ Errores comunes
* **Confundir token con palabra** (o con carácter): un token es lo que dice el vocabulario del tokenizer, y varía por modelo.
* **Comparar nDCG entre corpus o subconjuntos distintos** (o entre datos sintéticos y reales).
* **Calcular nDCG con `ranked_ids` sin ordenar** o con relevancias como booleanos si el dataset es graduado.
* **Mezclar `FAST_DEV_RUN=True` con conclusiones «de producción»**: el modo rápido es para depurar.
* **Olvidar fijar la semilla** y los empates: dos ejecuciones dan rankings distintos.
* **Dejar celdas Gemini sin reemplazar al compartir el notebook**: funcionan, pero verás el placeholder.

## 15. Autoevaluación

**1.** **(Cálculo)** Un ranking devuelve `[a, b, c, d, e]` y los relevantes son `{b: 1, e: 1}`. Calcula nDCG@5, Recall@3 y MRR@5.

<details>
<summary>Ver respuesta</summary>

DCG = $1/\log_2 3 + 1/\log_2 6 = 0.6309 + 0.3869 = 1.0178$. IDCG = $1 + 1/\log_2 3 = 1.6309$. **nDCG@5 = 0.624**. Recall@3: sólo `b` está en el top-3 → $1/2 = 0.5$. MRR@5: primer relevante en la posición 2 → $0.5$.

</details>

**2.** **(Diagnóstico)** Tu BM25 devuelve score 0 para todos los documentos de una consulta en español sobre un corpus en inglés. ¿Es un bug del índice? ¿Qué harías?

<details>
<summary>Ver respuesta</summary>

No es un bug: BM25 compara cadenas y la consulta no comparte ningún término con los documentos (IDF·TF = 0). Soluciones: un embedder multilingüe (módulos 10–12), traducir la consulta, o expandirla; y en tu evaluación, reportar el rendimiento *por idioma* para no promediar el problema.

</details>

**3.** **(Transferencia)** Tu golden set tiene consultas con 3 documentos relevantes con grados 3, 2 y 1. ¿Qué cambia en tu implementación y por qué la ganancia lineal puede ser discutible?

<details>
<summary>Ver respuesta</summary>

El IDCG se calcula con los 3 grados ordenados descendentemente (3, 2, 1) y el nDCG pasa a premiar el orden entre relevantes. La ganancia lineal ($\text{rel}$) es la de `trec_eval`; la exponencial ($2^{\text{rel}}-1$) enfatiza más los muy relevantes. Elegir una u otra cambia los números: fija la convención y no compares con trabajos que usan la otra.

</details>

**4.** ¿Por qué se usa el coseno (o el producto punto con vectores normalizados) y no la distancia euclídea sin normalizar para comparar embeddings de texto?

<details>
<summary>Ver respuesta</summary>

El coseno ignora la norma, que en muchos modelos refleja la frecuencia del token o la longitud del texto más que su significado. Con vectores normalizados, coseno, producto punto y distancia euclídea son equivalentes en ranking ($\lVert u-v\rVert^2 = 2-2\cos$).

</details>

**5.** ¿Qué dos hiperparámetros de BM25 controlan la saturación y la normalización por longitud, y qué valores toman con `b = 0`?

<details>
<summary>Ver respuesta</summary>

$k_1$ controla la saturación de la frecuencia y $b$ la normalización por longitud. Con $b=0$ no se normaliza: un documento largo no se penaliza por serlo.

</details>

**6.** Menciona dos razones por las que el español puede costar más tokens que el inglés con el tokenizer de GPT-2.

<details>
<summary>Ver respuesta</summary>

1) El vocabulario se construyó con mayoría de texto en inglés, así que hay menos fusiones para el español. 2) Las vocales con tilde y la «ñ» ocupan 2 bytes y, si no hay token que las contenga, se parten en bytes. (Lo desarrollan los módulos 01, 02 y 04.)

</details>

> **➡️ Siguiente módulo.** **01 · Texto como bytes.** Hemos visto que «ñ» pesa 2 bytes y que el español paga un sobrecoste. En el módulo 01 implementas UTF-8 a mano, distingues code points, bytes y grafemas, normalizas (NFC/NFD/NFKC) y auditas tu corpus Políglota: la capa sobre la que se construye cualquier tokenizer.

## 16. Referencias
Todas las referencias de arXiv fueron comprobadas (título y autores) el 25-oct-2026 y de nuevo, con venue y enlaces, el 26-oct-2026 (verificador de referencias).

* Kaplan, J. et al. (2020). *Scaling Laws for Neural Language Models.* arXiv:2001.08361. https://arxiv.org/abs/2001.08361 (aproximación de FLOPs por token, sección 2.1)
* Thakur, N., Reimers, N., Rücklé, A., Srivastava, A., Gurevych, I. (2021). *BEIR: A Heterogenous Benchmark for Zero-shot Evaluation of Information Retrieval Models.* NeurIPS 2021 (Datasets and Benchmarks). arXiv:2104.08663. https://arxiv.org/abs/2104.08663
* Robertson, S., Zaragoza, H. (2009). *The Probabilistic Relevance Framework: BM25 and Beyond.* Foundations and Trends in Information Retrieval 3(4).
* Reimers, N., Gurevych, I. (2019). *Sentence-BERT: Sentence Embeddings using Siamese BERT-Networks.* EMNLP-IJCNLP 2019. arXiv:1908.10084. https://arxiv.org/abs/1908.10084
* Radford, A. et al. (2019). *Language Models are Unsupervised Multitask Learners* (GPT-2). OpenAI.
* Wang, C., Cho, K., Gu, J. (2019). *Neural Machine Translation with Byte-Level Subwords.* arXiv:1909.03341. https://arxiv.org/abs/1909.03341
* Petrov, A., La Malfa, E., Torr, P., Bibi, A. (2023). *Language Model Tokenizers Introduce Unfairness Between Languages.* NeurIPS 2023. arXiv:2305.15425. https://arxiv.org/abs/2305.15425
* Ahia, O. et al. (2023). *Do All Languages Cost the Same? Tokenization in the Era of Commercial Language Models.* EMNLP 2023. arXiv:2305.13707. https://arxiv.org/abs/2305.13707
* Khattab, O., Zaharia, M. (2020). *ColBERT: Efficient and Effective Passage Search via Contextualized Late Interaction over BERT.* SIGIR 2020. arXiv:2004.12832. https://arxiv.org/abs/2004.12832
* Formal, T., Piwowarski, B., Clinchant, S. (2021). *SPLADE: Sparse Lexical and Expansion Model for First Stage Ranking.* SIGIR 2021 (short). arXiv:2107.05720. https://arxiv.org/abs/2107.05720
* Pagnoni, A. et al. (2024). *Byte Latent Transformer: Patches Scale Better Than Tokens.* arXiv:2412.09871. https://arxiv.org/abs/2412.09871
* Anthropic (19-sep-2024). *Introducing Contextual Retrieval.* https://www.anthropic.com/news/contextual-retrieval (con BM25 contextual + embeddings contextuales, los fallos de recuperación en el top-20 bajan un 49 %, del 5.7 % al 2.9 %, en su benchmark interno)
* Código: `tiktoken` (https://github.com/openai/tiktoken) · `rank_bm25` (https://pypi.org/project/rank-bm25/) · Hugging Face `wikimedia/wikipedia` (https://huggingface.co/datasets/wikimedia/wikipedia)